# Waymo Crash Data - Comprehensive Exploratory Analysis

This notebook provides a comprehensive analysis of Waymo autonomous vehicle crash data, including:

**Part 1: Interactive Visualization**
- Interactive Leaflet maps showing crash locations
- Heat maps for crash density
- Color-coded by crash type and severity

**Part 2: Temporal Analysis**
- Monthly/yearly trends
- Day of week patterns
- Seasonal variations

**Part 3: Spatial Analysis**
- Geographic distribution by city
- Crash type distribution
- Hotspot identification

**Part 4: Severity & Correlation Analysis**
- Severity indicators breakdown
- Correlation between time/location and severity

## Setup and Data Loading

In [ ]:
# Install required packages if needed
# !pip install pandas numpy matplotlib seaborn folium plotly

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)

print('Libraries loaded successfully!')

In [ ]:
# Load the data
DATA_PATH = '/Users/kateli/Desktop/Classes/COMM277T/bna-fuhgeddaboudit/cleaned-data/waymo_crashes_location.csv'
df = pd.read_csv(DATA_PATH)

# Rename columns for easier handling
col_mapping = {
    'SGO Report ID': 'report_id',
    'SGO Report Version': 'report_version',
    'SGO Amendment': 'amendment',
    'Year Month': 'year_month',
    'Location': 'city',
    'Crash Type': 'crash_type',
    'Is NHTSA Reportable In-Transport': 'nhtsa_reportable',
    'Is NHTSA Reportable In-Transport Delta-V Less than 1 MPH': 'low_severity',
    'Is Police-Reported': 'police_reported',
    'Is Any-Injury-Reported': 'injury_reported',
    'Is Any Vehicle Airbag Deployment': 'airbag_any',
    'Is Ego Vehicle Airbag Deployment': 'airbag_ego',
    'Is Suspected Serious Injury+': 'serious_injury',
    'Incident Date': 'incident_date',
    'Location Address / Description': 'address',
    'Zip Code': 'zip_code'
}
df = df.rename(columns=col_mapping)

# Parse dates and create temporal features
df['incident_date'] = pd.to_datetime(df['incident_date'], format='%m/%d/%Y', errors='coerce')
df['year'] = df['incident_date'].dt.year
df['month'] = df['incident_date'].dt.month
df['month_name'] = df['incident_date'].dt.month_name()
df['day_of_week'] = df['incident_date'].dt.day_name()
df['day_of_week_num'] = df['incident_date'].dt.dayofweek
df['is_weekend'] = df['day_of_week_num'].isin([5, 6])
df['quarter'] = df['incident_date'].dt.quarter
df['year_month_dt'] = df['incident_date'].dt.to_period('M')

# Clean city names
df['city_clean'] = df['city'].str.replace('_', ' ').str.title()

# Convert boolean columns
bool_cols = ['nhtsa_reportable', 'low_severity', 'police_reported',
             'injury_reported', 'airbag_any', 'airbag_ego', 'serious_injury']
for col in bool_cols:
    df[col] = df[col].map({'True': True, 'False': False, True: True, False: False})

# Create severity score
df['severity_score'] = (
    df['police_reported'].astype(int) * 1 +
    df['injury_reported'].astype(int) * 2 +
    df['airbag_any'].astype(int) * 2 +
    df['serious_injury'].astype(int) * 3
)

print(f'Total crashes: {len(df)}')
print(f'Date range: {df["incident_date"].min().date()} to {df["incident_date"].max().date()}')
print(f'Cities: {df["city_clean"].unique().tolist()}')
df.head()

---
# Part 1: Interactive Visualization
---

In [ ]:
# City center coordinates
CITY_COORDS = {
    'San Francisco': (37.7749, -122.4194),
    'Los Angeles': (34.0522, -118.2437),
    'Phoenix': (33.4484, -112.0740),
    'Austin': (30.2672, -97.7431),
    'Atlanta': (33.7490, -84.3880),
    'Mountain View': (37.3861, -122.0839)
}

# Sample of zip code coordinates (major areas)
ZIP_COORDS = {
    '94102': (37.7815, -122.4160), '94103': (37.7726, -122.4099),
    '94107': (37.7654, -122.3958), '94109': (37.7917, -122.4217),
    '94110': (37.7489, -122.4150), '94114': (37.7589, -122.4354),
    '94117': (37.7699, -122.4425), '94133': (37.8002, -122.4104),
    '85281': (33.4230, -111.9390), '85282': (33.3920, -111.9400),
    '85004': (33.4482, -112.0680), '85008': (33.4692, -111.9984),
    '85034': (33.4328, -112.0274), '85251': (33.4930, -111.9260),
    '90028': (34.0990, -118.3269), '90017': (34.0538, -118.2660),
    '78701': (30.2703, -97.7419), '78702': (30.2632, -97.7201),
    '30318': (33.7920, -84.4370), '30303': (33.7532, -84.3897)
}

def get_coordinates(row):
    zip_code = str(row['zip_code']).strip() if pd.notna(row['zip_code']) else None
    city = row['city_clean']
    
    if zip_code and zip_code in ZIP_COORDS:
        lat, lon = ZIP_COORDS[zip_code]
        lat += np.random.uniform(-0.005, 0.005)
        lon += np.random.uniform(-0.005, 0.005)
        return lat, lon
    
    if city in CITY_COORDS:
        lat, lon = CITY_COORDS[city]
        lat += np.random.uniform(-0.02, 0.02)
        lon += np.random.uniform(-0.02, 0.02)
        return lat, lon
    
    return None, None

coords = df.apply(get_coordinates, axis=1)
df['latitude'] = coords.apply(lambda x: x[0])
df['longitude'] = coords.apply(lambda x: x[1])

print(f'Geocoded {df["latitude"].notna().sum()} of {len(df)} crashes')

In [ ]:
import folium
from folium.plugins import MarkerCluster, HeatMap

# Color mapping for crash types
crash_colors = {
    'V2V F2R': 'blue', 'V2V Lateral': 'green', 'V2V Head-on': 'red',
    'V2V Backing': 'orange', 'V2V Intersection': 'purple',
    'Single Vehicle': 'gray', 'Pedestrian': 'darkred',
    'Cyclist': 'darkgreen', 'Motorcycle': 'darkblue', 'All Others': 'black'
}

# Create main map
df_coords = df[df['latitude'].notna()].copy()
m = folium.Map(location=[df_coords['latitude'].mean(), df_coords['longitude'].mean()], 
               zoom_start=5, tiles='OpenStreetMap')

# Add marker cluster
cluster = MarkerCluster(name='Crashes').add_to(m)

for _, row in df_coords.iterrows():
    color = crash_colors.get(row['crash_type'], 'gray')
    popup = f"""<b>Date:</b> {row['incident_date'].strftime('%Y-%m-%d') if pd.notna(row['incident_date']) else 'N/A'}<br>
    <b>City:</b> {row['city_clean']}<br>
    <b>Type:</b> {row['crash_type']}<br>
    <b>Address:</b> {row['address']}<br>
    <b>Severity:</b> {row['severity_score']}/8"""
    
    folium.CircleMarker(
        [row['latitude'], row['longitude']],
        radius=5 + row['severity_score'] * 2,
        color=color, fill=True, fillColor=color, fillOpacity=0.6,
        popup=popup
    ).add_to(cluster)

# Add heatmap
HeatMap(df_coords[['latitude', 'longitude']].values.tolist(), 
        name='Density', radius=15).add_to(m)

folium.LayerControl().add_to(m)
m

### City-Specific Maps

In [ ]:
# San Francisco map
sf_df = df_coords[df_coords['city_clean'] == 'San Francisco']
sf_map = folium.Map(location=[37.7749, -122.4194], zoom_start=12)

for _, row in sf_df.iterrows():
    color = crash_colors.get(row['crash_type'], 'gray')
    folium.CircleMarker(
        [row['latitude'], row['longitude']],
        radius=5, color=color, fill=True, fillOpacity=0.6
    ).add_to(sf_map)

HeatMap(sf_df[['latitude', 'longitude']].values.tolist(), radius=12).add_to(sf_map)
print(f'San Francisco: {len(sf_df)} crashes')
sf_map

In [ ]:
# Phoenix map
phx_df = df_coords[df_coords['city_clean'] == 'Phoenix']
phx_map = folium.Map(location=[33.4484, -112.0740], zoom_start=11)

for _, row in phx_df.iterrows():
    color = crash_colors.get(row['crash_type'], 'gray')
    folium.CircleMarker(
        [row['latitude'], row['longitude']],
        radius=5, color=color, fill=True, fillOpacity=0.6
    ).add_to(phx_map)

HeatMap(phx_df[['latitude', 'longitude']].values.tolist(), radius=12).add_to(phx_map)
print(f'Phoenix: {len(phx_df)} crashes')
phx_map

---
# Part 2: Temporal Analysis
---

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Monthly trend
monthly = df.groupby('year_month_dt').size()
monthly.plot(ax=axes[0,0], marker='o', linewidth=2, color='steelblue')
axes[0,0].set_title('Monthly Crash Trend', fontweight='bold')
axes[0,0].set_xlabel('Month')
axes[0,0].set_ylabel('Crashes')
axes[0,0].tick_params(axis='x', rotation=45)

# 2. Day of week
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_counts = df['day_of_week'].value_counts().reindex(day_order)
colors = ['steelblue']*5 + ['coral']*2
axes[0,1].bar(range(7), day_counts.values, color=colors, edgecolor='black')
axes[0,1].set_xticks(range(7))
axes[0,1].set_xticklabels([d[:3] for d in day_order])
axes[0,1].set_title('Crashes by Day of Week', fontweight='bold')
axes[0,1].set_ylabel('Crashes')

# 3. Monthly distribution
month_order = ['January', 'February', 'March', 'April', 'May', 'June',
               'July', 'August', 'September', 'October', 'November', 'December']
month_counts = df['month_name'].value_counts().reindex(month_order).dropna()
axes[1,0].bar(range(len(month_counts)), month_counts.values, color='teal', edgecolor='black')
axes[1,0].set_xticks(range(len(month_counts)))
axes[1,0].set_xticklabels([m[:3] for m in month_counts.index], rotation=45)
axes[1,0].set_title('Crashes by Month', fontweight='bold')
axes[1,0].set_ylabel('Crashes')

# 4. Year-over-year
yearly = df.groupby(['year', 'month']).size().reset_index(name='count')
for year in yearly['year'].unique():
    yd = yearly[yearly['year'] == year]
    axes[1,1].plot(yd['month'], yd['count'], marker='o', label=str(year))
axes[1,1].set_title('Year-over-Year Comparison', fontweight='bold')
axes[1,1].legend(title='Year')
axes[1,1].set_xlabel('Month')
axes[1,1].set_ylabel('Crashes')

plt.tight_layout()
plt.show()

# Summary stats
print('\n--- TEMPORAL SUMMARY ---')
print(f'Busiest day: {day_counts.idxmax()} ({day_counts.max()} crashes)')
print(f'Quietest day: {day_counts.idxmin()} ({day_counts.min()} crashes)')
print(f'Weekday crashes: {df[~df["is_weekend"]].shape[0]} ({df[~df["is_weekend"]].shape[0]/len(df)*100:.1f}%)')
print(f'Weekend crashes: {df[df["is_weekend"]].shape[0]} ({df[df["is_weekend"]].shape[0]/len(df)*100:.1f}%)')

---
# Part 3: Spatial Analysis
---

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. City distribution
city_counts = df['city_clean'].value_counts()
axes[0,0].barh(range(len(city_counts)), city_counts.values, color=plt.cm.Set2(range(len(city_counts))))
axes[0,0].set_yticks(range(len(city_counts)))
axes[0,0].set_yticklabels(city_counts.index)
axes[0,0].set_title('Crashes by City', fontweight='bold')
axes[0,0].set_xlabel('Number of Crashes')
for i, v in enumerate(city_counts.values):
    axes[0,0].text(v + 5, i, f'{v} ({v/len(df)*100:.1f}%)', va='center')

# 2. Crash type distribution
type_counts = df['crash_type'].value_counts()
axes[0,1].barh(range(len(type_counts)), type_counts.values, color='teal', alpha=0.7)
axes[0,1].set_yticks(range(len(type_counts)))
axes[0,1].set_yticklabels(type_counts.index)
axes[0,1].set_title('Crashes by Type', fontweight='bold')
axes[0,1].set_xlabel('Number of Crashes')

# 3. Crash type x city heatmap
ct_city = pd.crosstab(df['crash_type'], df['city_clean'])
sns.heatmap(ct_city, annot=True, fmt='d', cmap='YlOrRd', ax=axes[1,0])
axes[1,0].set_title('Crash Type × City', fontweight='bold')

# 4. City pie chart
axes[1,1].pie(city_counts.values, labels=city_counts.index, autopct='%1.1f%%',
              colors=plt.cm.Set3(range(len(city_counts))))
axes[1,1].set_title('Geographic Distribution', fontweight='bold')

plt.tight_layout()
plt.show()

print('\n--- SPATIAL SUMMARY ---')
for city, count in city_counts.items():
    print(f'{city}: {count} ({count/len(df)*100:.1f}%)')

In [ ]:
# Geographic hotspots (top zip codes)
zip_counts = df['zip_code'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(range(len(zip_counts)), zip_counts.values, color=plt.cm.tab20(range(len(zip_counts))))
ax.set_yticks(range(len(zip_counts)))
labels = [f"{z} ({df[df['zip_code']==z]['city_clean'].mode()[0][:8] if len(df[df['zip_code']==z]) > 0 else '?'})" 
          for z in zip_counts.index]
ax.set_yticklabels(labels)
ax.set_title('Top 15 Crash Hotspots (by Zip Code)', fontweight='bold', fontsize=14)
ax.set_xlabel('Number of Crashes')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print('\n--- TOP HOTSPOTS ---')
for i, (z, c) in enumerate(zip_counts.head(10).items(), 1):
    city = df[df['zip_code']==z]['city_clean'].mode()[0] if len(df[df['zip_code']==z]) > 0 else 'Unknown'
    print(f'{i}. {z} ({city}): {c} crashes ({c/len(df)*100:.1f}%)')

---
# Part 4: Severity & Correlation Analysis
---

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Severity indicators
sev_cols = ['police_reported', 'injury_reported', 'airbag_any', 'serious_injury']
sev_counts = df[sev_cols].sum()
colors_sev = ['#ff6b6b', '#feca57', '#48dbfb', '#ff9ff3']
bars = axes[0,0].bar(range(4), sev_counts.values, color=colors_sev, edgecolor='black')
axes[0,0].set_xticks(range(4))
axes[0,0].set_xticklabels(['Police\nReported', 'Injury\nReported', 'Airbag\nDeployed', 'Serious\nInjury'])
axes[0,0].set_title('Severity Indicators', fontweight='bold')
for i, v in enumerate(sev_counts.values):
    axes[0,0].text(i, v+3, f'{v}\n({v/len(df)*100:.1f}%)', ha='center')

# 2. Severity score distribution
sev_dist = df['severity_score'].value_counts().sort_index()
axes[0,1].bar(sev_dist.index, sev_dist.values, color='darkred', alpha=0.7, edgecolor='black')
axes[0,1].set_xlabel('Severity Score')
axes[0,1].set_ylabel('Number of Crashes')
axes[0,1].set_title('Severity Score Distribution', fontweight='bold')

# 3. Severity by city
city_sev = df.groupby('city_clean')['severity_score'].mean().sort_values(ascending=False)
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(city_sev)))
axes[1,0].barh(range(len(city_sev)), city_sev.values, color=colors)
axes[1,0].set_yticks(range(len(city_sev)))
axes[1,0].set_yticklabels(city_sev.index)
axes[1,0].set_title('Average Severity by City', fontweight='bold')
axes[1,0].set_xlabel('Avg Severity Score')

# 4. Severity by crash type
type_sev = df.groupby('crash_type')['severity_score'].mean().sort_values(ascending=False)
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(type_sev)))
axes[1,1].barh(range(len(type_sev)), type_sev.values, color=colors)
axes[1,1].set_yticks(range(len(type_sev)))
axes[1,1].set_yticklabels(type_sev.index, fontsize=9)
axes[1,1].set_title('Average Severity by Crash Type', fontweight='bold')
axes[1,1].set_xlabel('Avg Severity Score')

plt.tight_layout()
plt.show()

print('\n--- SEVERITY SUMMARY ---')
print(f'Average severity score: {df["severity_score"].mean():.2f}/8')
print(f'Most severe city: {city_sev.index[0]} ({city_sev.values[0]:.2f})')
print(f'Most severe crash type: {type_sev.index[0]} ({type_sev.values[0]:.2f})')

In [ ]:
# Correlation: Time × Location × Severity
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Day x City heatmap
dow_city = pd.crosstab(df['day_of_week'], df['city_clean'])
dow_city = dow_city.reindex(day_order)
sns.heatmap(dow_city, annot=True, fmt='d', cmap='Blues', ax=axes[0,0])
axes[0,0].set_title('Day of Week × City', fontweight='bold')

# 2. Month x City heatmap
month_city = pd.crosstab(df['month_name'], df['city_clean'])
month_city = month_city.reindex([m for m in month_order if m in month_city.index])
sns.heatmap(month_city, annot=True, fmt='d', cmap='Greens', ax=axes[0,1])
axes[0,1].set_title('Month × City', fontweight='bold')

# 3. Severity by day of week
dow_sev = df.groupby('day_of_week')['severity_score'].mean().reindex(day_order)
colors = ['steelblue']*5 + ['coral']*2
axes[1,0].bar(range(7), dow_sev.values, color=colors, edgecolor='black')
axes[1,0].set_xticks(range(7))
axes[1,0].set_xticklabels([d[:3] for d in day_order])
axes[1,0].set_title('Avg Severity by Day of Week', fontweight='bold')
axes[1,0].axhline(df['severity_score'].mean(), color='red', linestyle='--', label='Overall Avg')
axes[1,0].legend()

# 4. Severity by month
month_sev = df.groupby('month_name')['severity_score'].mean()
month_sev = month_sev.reindex([m for m in month_order if m in month_sev.index])
axes[1,1].plot(range(len(month_sev)), month_sev.values, marker='o', linewidth=2, color='darkred')
axes[1,1].fill_between(range(len(month_sev)), month_sev.values, alpha=0.3, color='darkred')
axes[1,1].set_xticks(range(len(month_sev)))
axes[1,1].set_xticklabels([m[:3] for m in month_sev.index])
axes[1,1].set_title('Avg Severity by Month', fontweight='bold')
axes[1,1].axhline(df['severity_score'].mean(), color='blue', linestyle='--', label='Overall Avg')
axes[1,1].legend()

plt.tight_layout()
plt.show()

---
# Summary Statistics
---

In [ ]:
print('='*60)
print('WAYMO CRASH DATA - COMPREHENSIVE SUMMARY')
print('='*60)

print(f'\nTotal Crashes: {len(df)}')
print(f'Date Range: {df["incident_date"].min().date()} to {df["incident_date"].max().date()}')
print(f'Cities: {len(df["city_clean"].unique())}')
print(f'Crash Types: {df["crash_type"].nunique()}')
print(f'Unique Zip Codes: {df["zip_code"].nunique()}')

print('\n--- Severity Rates ---')
print(f'Police Reported: {df["police_reported"].sum()} ({df["police_reported"].sum()/len(df)*100:.1f}%)')
print(f'Injury Reported: {df["injury_reported"].sum()} ({df["injury_reported"].sum()/len(df)*100:.1f}%)')
print(f'Airbag Deployed: {df["airbag_any"].sum()} ({df["airbag_any"].sum()/len(df)*100:.1f}%)')
print(f'Serious Injury: {df["serious_injury"].sum()} ({df["serious_injury"].sum()/len(df)*100:.1f}%)')
print(f'\nAvg Severity Score: {df["severity_score"].mean():.2f}/8')

print('\n--- Key Findings ---')
print(f'1. Most crashes in: {city_counts.index[0]} ({city_counts.values[0]} = {city_counts.values[0]/len(df)*100:.1f}%)')
print(f'2. Most common type: {type_counts.index[0]} ({type_counts.values[0]} = {type_counts.values[0]/len(df)*100:.1f}%)')
print(f'3. Busiest day: {day_counts.idxmax()} ({day_counts.max()} crashes)')
print(f'4. Most severe city: {city_sev.index[0]} (avg score: {city_sev.values[0]:.2f})')
print(f'5. Most severe type: {type_sev.index[0]} (avg score: {type_sev.values[0]:.2f})')

---
## Output Files

The standalone Python script (`waymo_comprehensive_analysis.py`) generates:
- `waymo_crashes_interactive_map.html` - Main interactive map with all crashes
- `waymo_crashes_[city]_map.html` - City-specific interactive maps
- `temporal_analysis.png` - Temporal analysis charts
- `spatial_analysis.png` - Spatial analysis charts
- `severity_analysis.png` - Severity analysis charts
- `correlation_analysis.png` - Correlation analysis charts
- `geographic_clustering.png` - Hotspot analysis
- `waymo_crashes_processed.csv` - Processed data with added features